# 강화학습 3강 — 기억하기 (리플레이 메모리)

2강에서 두뇌는 한 걸음 걸을 때마다 **딱 한 번 배우고 그 경험을 버렸습니다.**
그래서 보물을 한 번 봐도, 그 기억이 출발점까지 전해지기 전에 잊어버렸죠.

오늘은 겪은 일을 **기억 상자(리플레이 메모리)** 에 쌓아 두고, 무작위로 꺼내 **여러 번 다시 배웁니다.**
무대는 Gymnasium 의 **얼음 호수(FrozenLake 8×8)** — 64칸 중 목표는 단 하나, 보상도 거기서만 나옵니다.

| 셀 | 하는 일 |
|---|---|
| 1 | 준비 |
| 2 | 얼음 호수 열기 |
| 3 | 행동가치 Q(s, a) — 두뇌가 외우는 숫자 |
| 4 | SARS — 경험 한 조각 |
| 5 | 기억 상자 만들기 (채우는 법 · 꺼내는 법) |
| 6 | 학습 함수 — SARS 를 버릴까, 쌓을까 |
| 7 | ① 쌓지 않고(2강) vs ② 쌓아서 무작위로 꺼내기 |
| 8 | 학습 곡선 · 출발점의 Q 값이 자라는 모습 |
| 9 | 🎬 학습되는 과정 — 가치가 퍼지는 애니메이션 |
| 10 | 🎬 다 배운 두뇌가 걷는 영상 |
| 11 | ⚔️ 꺼내는 방법 5가지 대결 (씨앗 5개) |
| 12 | 기억 상자 실험 — 크기와 꺼내는 수 |
| 13 | 🌗 좋은 기억만 · 나쁜 기억만 |
| 14 | 🪞 기억을 내 맘대로 바꾸면? (왜곡) |

> 위에서부터 순서대로 ▶ 실행하세요. GPU 는 필요 없습니다. (런타임 → 모두 실행 도 됩니다)

## 1. 준비

In [ ]:
!pip install -q "gymnasium[toy-text]"

import random
from collections import deque
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

SEED = 0          # 같은 숫자면 언제 실행해도 같은 결과
학습률 = 0.5       # 새 경험을 얼마나 크게 반영할까 (2강과 같음)
할인율 = 0.95      # 미래 보상을 얼마나 쳐줄까 (2강과 같음)
plt.rcParams["figure.dpi"] = 110
print("준비 완료 · gymnasium", gym.__version__)

## 2. 얼음 호수 열기

- **S** 출발 · **F** 얼음(걸어도 됨) · **H** 구멍(빠지면 끝, 0점) · **G** 목표(도착하면 **1점**, 끝)
- 행동은 4개: 0 = ←, 1 = ↓, 2 = →, 3 = ↑
- 보상은 **G 에서만** 나옵니다. 무작위로 걸어서 G 에 닿는 일은 아주 드뭅니다.

In [ ]:
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False, render_mode="rgb_array")
env.reset(seed=SEED)
지도 = env.unwrapped.desc.astype(str)
칸수 = env.observation_space.n
print("칸 수:", 칸수, "· 행동 수:", env.action_space.n)
print("\n".join(" ".join(줄) for 줄 in 지도))

plt.figure(figsize=(4, 4))
plt.imshow(env.render())
plt.axis("off")
plt.title("FrozenLake 8x8")
plt.show()

# 무작위로 1000판 걸어 보면 G 에 몇 번 닿을까?
닿은_판 = 0
for i in range(1000):
    s, _ = env.reset()
    끝 = False
    while not 끝:
        s, r, terminated, truncated, _ = env.step(env.action_space.sample())
        끝 = terminated or truncated
    닿은_판 += r
print(f"무작위 1000판 중 목표 도착: {닿은_판:.0f}판")

## 3. 행동가치 Q(s, a) — 두뇌가 외우는 숫자

**Q(s, a)** = 「칸 **s** 에서 행동 **a** 를 하면, 앞으로 받을 보상이 얼마나 될까?」 에 대한 두뇌의 추측입니다.

- 칸 64개 × 행동 4개 = **숫자 256개** 짜리 표. 처음엔 전부 0 (아무것도 모름)
- 두뇌는 표에서 **가장 큰 숫자의 행동**을 고릅니다 (같으면 아무거나)
- 배울 때는 이렇게 고칩니다 (2강과 같은 식):

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,\big]$$

「받은 보상 r + 다음 칸에서 기대되는 최고 가치(할인)」 쪽으로 조금(α) 다가갑니다.
**G 바로 옆 칸**에서 G 로 가면 1점을 받으니 그 칸의 Q 가 먼저 오르고, 그 값이 한 칸씩 **거꾸로** 출발점까지 번져 갑니다.

In [ ]:
화살표 = ["←", "↓", "→", "↑"]

def 행동_고르기(Q, s, 엡실론, rng):
    """엡실론 확률로 무작위 탐험, 아니면 가장 큰 Q (같으면 아무거나)"""
    if rng.random() < 엡실론:
        return rng.randrange(4)
    최고 = Q[s].max()
    return rng.choice([a for a in range(4) if Q[s, a] == 최고])

def Q_고치기(Q, s, a, r, s2, 끝):
    목표값 = r if 끝 else r + 할인율 * Q[s2].max()
    Q[s, a] += 학습률 * (목표값 - Q[s, a])

# 예시: G 바로 왼쪽 칸(62)에서 → 를 해서 G(63)에 도착, 1점
Q = np.zeros((칸수, 4))
print("고치기 전 Q[62] =", Q[62])
Q_고치기(Q, 62, 2, 1.0, 63, True)
print("고친 뒤  Q[62] =", Q[62], "← 이제 62번 칸에서 →(2)가 좋다는 걸 압니다")
Q_고치기(Q, 54, 1, 0.0, 62, False)
print("그 위 칸 Q[54] =", Q[54].round(3), "← 보상은 0인데도, 다음 칸(62)이 좋으니 ↓(1)가 조금 올랐습니다")

## 4. SARS — 경험 한 조각

한 걸음 걸을 때마다 경험 한 조각이 생깁니다: **S**(지금 칸) · **A**(한 행동) · **R**(받은 보상) · **S'**(다음 칸) + 끝났는지.
2강의 두뇌는 이 조각으로 Q 를 **한 번 고치고 버렸습니다.** 오늘은 이 조각을 **쌓아 둡니다.**

In [ ]:
env2 = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
s, _ = env2.reset()
for a in [2, 2, 1]:                       # → → ↓ 세 걸음
    s2, r, terminated, truncated, _ = env2.step(a)
    print(f"SARS = (S={s:2}, A={화살표[a]}, R={r}, S'={s2:2}, 끝={terminated})")
    s = s2
env2.close()
print("\n보상 R 은 G 에 도착할 때만 1 입니다. 대부분의 조각은 R=0 이에요.")

## 5. 기억 상자 만들기 — 채우는 법 · 꺼내는 법

같은 상자라도 **어떻게 채우고 어떻게 꺼내느냐**에 따라 배우는 게 달라집니다.

| 채우는 법 | 뜻 |
|---|---|
| `오래된것_지우기` | 가득 차면 가장 오래된 기억을 지우고 새 기억을 넣는다 (기본) |
| `처음것_유지` | 가득 차면 더 넣지 않는다 — 처음에 겪은 것들로만 계속 배운다 |

| 꺼내는 법 | 뜻 |
|---|---|
| `무작위` | 상자 전체에서 무작위로 (기본) |
| `최근` | 가장 최근 기억부터 순서대로 |
| `보상우선` | 보상을 받은 기억(G 도착)을 일부러 더 자주 섞는다 |

In [ ]:
class 기억상자:
    def __init__(self, 크기, 채우는법="오래된것_지우기", 꺼내는법="무작위"):
        self.크기, self.채우는법, self.꺼내는법 = 크기, 채우는법, 꺼내는법
        self.상자 = deque(maxlen=크기 if 채우는법 == "오래된것_지우기" else None)

    def 넣기(self, s, a, r, s2, 끝):
        if self.채우는법 == "처음것_유지" and len(self.상자) >= self.크기:
            return                                   # 가득 찼으면 더 안 넣음
        self.상자.append((s, a, r, s2, 끝))

    def 꺼내기(self, 몇개, rng):
        if self.꺼내는법 == "최근":
            return list(self.상자)[-몇개:]
        if self.꺼내는법 == "보상우선":
            좋은기억 = [m for m in self.상자 if m[2] > 0]
            n = min(len(좋은기억), 몇개 // 4)          # 꺼내는 수의 1/4 까지는 G 도착 기억으로
            return rng.sample(좋은기억, n) + rng.sample(self.상자, 몇개 - n)
        return rng.sample(self.상자, 몇개)            # 무작위

    def __len__(self):
        return len(self.상자)

# 써 보기 — 크기 5 상자에 SARS 7개를 넣으면?
for 채우는법 in ["오래된것_지우기", "처음것_유지"]:
    상자 = 기억상자(5, 채우는법=채우는법)
    for i in range(7):
        상자.넣기(i, 2, 0.0, i + 1, False)
    print(f"{채우는법:9} → 남은 S:", [m[0] for m in 상자.상자])

## 6. 학습 함수 — SARS 를 버릴까, 쌓을까

`기억_상자=None` 이면 2강 방식(한 번 배우고 버림). 상자를 주면: 걸을 때마다 SARS 를 넣고, `꺼낼_수` 만큼 꺼내 Q 를 다시 고칩니다.
학습되는 과정을 영상으로 보려고 10판마다 Q 표 사진을 찍고, **출발점의 Q 값**도 매 판 기록합니다.

In [ ]:
def 학습(판수=500, 기억_상자=None, 꺼낼_수=32, seed=SEED, 사진_간격=10):
    rng = random.Random(seed)
    env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
    Q = np.zeros((칸수, 4))
    엡실론, 줄이기 = 1.0, 0.01 ** (1 / (판수 * 0.6))    # 판수의 60% 지점에서 1% 가 되도록 (2강의 엡실론-그리디)
    성공들, 출발점_Q, 사진들, 다시배운_횟수 = [], [], [], 0
    for 판 in range(판수):
        s, _ = env.reset()
        끝, 보상합 = False, 0.0
        while not 끝:
            a = 행동_고르기(Q, s, 엡실론, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            끝 = terminated or truncated
            보상합 += r
            if 기억_상자 is None:
                Q_고치기(Q, s, a, r, s2, terminated)            # 한 번 배우고 버림 (2강)
            else:
                기억_상자.넣기(s, a, r, s2, terminated)          # SARS 를 상자에 쌓기
                if len(기억_상자) >= 꺼낼_수:
                    for 기억 in 기억_상자.꺼내기(꺼낼_수, rng):    # 꺼내서 다시 배우기
                        Q_고치기(Q, *기억)
                        다시배운_횟수 += 1
            s = s2
        성공들.append(보상합)
        출발점_Q.append(Q[0].max())
        엡실론 = max(0.01, 엡실론 * 줄이기)
        if 판 % 사진_간격 == 0 or 판 == 판수 - 1:
            사진들.append((판 + 1, Q.copy()))
    env.close()
    return dict(Q=Q, 성공들=np.array(성공들), 출발점_Q=np.array(출발점_Q), 사진들=사진들, 다시배운_횟수=다시배운_횟수)

def 다_배운_두뇌_시험(Q):
    # 탐험 없이(가장 큰 Q 만) 걸어서 목표에 도착하나? (미끄럽지 않으니 한 번이면 충분)
    env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
    rng = random.Random(123)
    s, _ = env.reset()
    끝 = False
    while not 끝:
        s, r, terminated, truncated, _ = env.step(행동_고르기(Q, s, 0.0, rng))
        끝 = terminated or truncated
    env.close()
    return "도착 ✅" if r > 0 else "실패 ❌"
print("준비 완료")

## 7. ① 쌓지 않고(2강) vs ② 쌓아서 무작위로 꺼내기 — 같은 500판

두 두뇌는 **딱 한 가지만** 다릅니다: SARS 를 버리느냐, 기억 상자(크기 1만)에 쌓아 두고 걸음마다 32개를 무작위로 꺼내 다시 배우느냐.

In [ ]:
기억없이 = 학습(기억_상자=None)
기억으로 = 학습(기억_상자=기억상자(10000), 꺼낼_수=32)
print(f"기억에서 다시 배운 횟수: {기억으로['다시배운_횟수']:,}번 (기억 없이: 걸음마다 1번)")

### 결과 표

In [ ]:
def 처음_도착(성공들):
    i = np.flatnonzero(성공들 > 0)
    return f"{i[0] + 1}판째" if len(i) else "못 함"

print(f"{'':10} {'1~100':>7} {'101~200':>8} {'201~300':>8} {'301~400':>8} {'401~500':>8}   처음 도착   다 배운 두뇌")
for 이름, 결과 in [("기억 없이", 기억없이), ("기억 상자", 기억으로)]:
    구간 = [결과["성공들"][i:i + 100].mean() for i in range(0, 500, 100)]
    print(f"{이름:8} " + " ".join(f"{v:8.0%}" for v in 구간) + f"   {처음_도착(결과['성공들']):>8}   {다_배운_두뇌_시험(결과['Q']):>9}")
print("\n※ 표의 숫자 = 그 구간에서 목표에 도착한 판의 비율")

## 8. 학습 곡선 · 출발점의 Q 값이 자라는 모습

왼쪽: 목표에 도착한 비율. 오른쪽: **출발 칸 S 의 가장 큰 Q** — G 의 보상이 출발점까지 전해졌다는 신호입니다.
가장 짧은 길(14걸음)로 가면 이 값은 0.95¹³ ≈ **0.51** 에 가까워집니다.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for 이름, 결과, 색 in [("no memory (lesson 2)", 기억없이, "#999999"), ("replay memory", 기억으로, "#2ca02c")]:
    a1.plot(np.convolve(결과["성공들"], np.ones(25) / 25, mode="valid"), label=이름, color=색, lw=2.5)
    a2.plot(결과["출발점_Q"], label=이름, color=색, lw=2.5)
a1.set(xlabel="episode", ylabel="success rate (25-ep avg)", ylim=(-0.02, 1.02)); a1.legend(); a1.grid(alpha=.3)
a2.axhline(0.95 ** 13, ls="--", color="#ffb000", label="best path 0.95^13"); a2.set(xlabel="episode", ylabel="max Q at start S"); a2.legend(); a2.grid(alpha=.3)
plt.show()

## 9. 🎬 학습되는 과정 — 가치가 퍼지는 애니메이션

각 칸의 색 = 그 칸에서 **가장 큰 Q** (밝을수록 「여기서 G 까지 갈 수 있다」고 믿는 칸). 화살표 = 두뇌가 고를 행동.
G 에서 시작한 밝은 색이 **거꾸로 출발점까지 번져 가는 속도**를 비교해 보세요. ▶ 버튼으로 재생합니다.

In [ ]:
def 그리기(ax, Q, 제목):
    ax.clear()
    가치 = Q.max(axis=1).reshape(8, 8)
    ax.imshow(가치, cmap="viridis", vmin=0, vmax=1)
    for 칸 in range(칸수):
        y, x = divmod(칸, 8)
        글자 = 지도[y][x]
        if 글자 == "H":
            ax.add_patch(plt.Rectangle((x - .5, y - .5), 1, 1, color="#1b1b1b"))
            ax.text(x, y, "H", ha="center", va="center", color="#ff6b6b", fontsize=11, weight="bold")
        elif 글자 == "G":
            ax.text(x, y, "G", ha="center", va="center", color="black", fontsize=13, weight="bold",
                    bbox=dict(boxstyle="round", fc="#ffd447"))
        elif Q[칸].max() > 0:
            ax.text(x, y, 화살표[int(Q[칸].argmax())], ha="center", va="center", color="white", fontsize=12)
        if 글자 == "S":
            ax.text(x - .42, y - .3, "S", color="white", fontsize=8, weight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(제목, fontsize=11)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.6))
장면수 = min(len(기억없이["사진들"]), len(기억으로["사진들"]))
def 장면(i):
    판1, Q1 = 기억없이["사진들"][i]
    판2, Q2 = 기억으로["사진들"][i]
    그리기(ax1, Q1, f"no memory · episode {판1}")
    그리기(ax2, Q2, f"replay memory · episode {판2}")
영상 = animation.FuncAnimation(fig, 장면, frames=장면수, interval=220)
plt.close(fig)
HTML(영상.to_jshtml())

## 10. 🎬 다 배운 두뇌가 걷는 영상

탐험 없이(가장 큰 Q 만) 걷습니다. 왼쪽은 기억 없이, 오른쪽은 기억 상자로 배운 두뇌.

In [ ]:
def 걸음_영상_장면들(Q, 최대=40):
    env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False, render_mode="rgb_array")
    rng = random.Random(7)
    s, _ = env.reset()
    장면들, 끝, 결과 = [env.render()], False, "stuck"
    for _ in range(최대):
        s, r, terminated, truncated, _ = env.step(행동_고르기(Q, s, 0.0, rng))
        장면들.append(env.render())
        끝 = terminated or truncated
        if 끝:
            결과 = "GOAL!" if r > 0 else "fell in a hole"
            break
    env.close()
    return 장면들, 결과

왼쪽, 결과1 = 걸음_영상_장면들(기억없이["Q"])
오른쪽, 결과2 = 걸음_영상_장면들(기억으로["Q"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4.3))
그림1, 그림2 = ax1.imshow(왼쪽[0]), ax2.imshow(오른쪽[0])
for ax in (ax1, ax2): ax.axis("off")
def 걸음(i):
    그림1.set_data(왼쪽[min(i, len(왼쪽) - 1)])
    그림2.set_data(오른쪽[min(i, len(오른쪽) - 1)])
    ax1.set_title("no memory · " + (결과1 if i >= len(왼쪽) - 1 else f"step {i}"))
    ax2.set_title("replay memory · " + (결과2 if i >= len(오른쪽) - 1 else f"step {i}"))
영상 = animation.FuncAnimation(fig, 걸음, frames=max(len(왼쪽), len(오른쪽)) + 6, interval=260)
plt.close(fig)
HTML(영상.to_jshtml())

## 11. ⚔️ 꺼내는 방법 5가지 대결

SARS 를 **어떻게 쌓고 어떻게 꺼낼지** 다섯 가지를 같은 조건(500판, 꺼낼 수 32)으로 겨룹니다.
강화학습은 운(씨앗)에 따라 결과가 크게 달라서, **씨앗 5개로 각각 학습**해 평균을 냅니다. (1~2분 걸려요)

In [ ]:
방법들 = {
    "① 쌓지 않음 (2강)":        lambda: None,
    "② 무작위로 꺼내기":         lambda: 기억상자(10000, 꺼내는법="무작위"),
    "③ 최근 것만 꺼내기":        lambda: 기억상자(10000, 꺼내는법="최근"),
    "④ 처음 2000개만 채워 두기":  lambda: 기억상자(2000, 채우는법="처음것_유지"),
    "⑤ 보상 받은 기억 우선":      lambda: 기억상자(10000, 꺼내는법="보상우선"),
}
씨앗들 = [0, 1, 2, 3, 4]
대결 = {}
for 이름, 상자_만들기 in 방법들.items():
    결과들 = [학습(판수=500, 기억_상자=상자_만들기(), 꺼낼_수=32, seed=sd, 사진_간격=10_000) for sd in 씨앗들]
    대결[이름] = 결과들
    성공률 = np.mean([r["성공들"].mean() for r in 결과들])
    도착 = sum(다_배운_두뇌_시험(r["Q"]).startswith("도착") for r in 결과들)
    print(f"{이름:18} 성공률 {성공률:5.1%} · 다 배운 두뇌 도착 {도착}/5 · 씨앗별 {[round(float(r['성공들'].mean()), 2) for r in 결과들]}")

이름들 = list(대결)
평균 = [np.mean([r["성공들"].mean() for r in 대결[n]]) for n in 이름들]
plt.figure(figsize=(10, 3.8))
막대 = plt.barh(range(len(이름들))[::-1], 평균, color=["#999999", "#2ca02c", "#1f77b4", "#d62728", "#ff7f0e"])
plt.yticks(range(len(이름들))[::-1], ["1 no memory", "2 random", "3 recent only", "4 first 2000 only", "5 reward first"])
for b, v in zip(막대, 평균):
    plt.text(v + 0.005, b.get_y() + b.get_height() / 2, f"{v:.0%}", va="center")
plt.xlabel("success rate during 500 episodes (5 seeds avg)")
plt.xlim(0, max(평균) * 1.25 + 0.01)
plt.show()

# 씨앗 0 에서 각 방법이 만든 Q 지도 (밝을수록 G 까지 갈 수 있다고 믿는 칸)
fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, 이름, 짧은 in zip(axes, 이름들, ["1 no memory", "2 random", "3 recent", "4 first only", "5 reward first"]):
    그리기(ax, 대결[이름][0]["Q"], 짧은)
plt.show()

### 결과를 읽는 법

- **④ 처음 것만 채워 두기**가 거의 0% 인 이유: 처음 2000개의 SARS 는 탐험(엡실론 100%) 중의 무작위 걸음이라 **G 에 도착한 조각이 하나도 없습니다.** 상자에 없는 경험은 다시 배울 수도 없어요. → 기억은 **새로 채워져야** 합니다.
- **③ 최근 것만**이 이 실험에서 강한 이유: 방금 걸은 길을 거꾸로 여러 번 다시 배우니 G 의 가치가 그 길을 따라 빨리 번집니다. 표(Q 표)로 배울 때는 이게 잘 통해요.
  다만 **4강에서 표 대신 딥러닝을 쓰면 이야기가 달라집니다.** 그때 「왜 무작위로 꺼내야 하는지」 다시 확인합니다.
- **씨앗별 숫자가 크게 다른 것**도 꼭 보세요. 한 번 돌려서 이긴 방법이 진짜 좋은 방법이라고 말할 수 없습니다.

## 12. 기억 상자 실험 — 크기와 꺼내는 수

상자가 너무 작으면 귀한 기억이 금방 지워지고, 꺼내는 수가 많으면 계산이 오래 걸립니다. 300판씩 숫자를 바꿔 비교해 보세요.

In [ ]:
상자_크기들 = [100, 1000, 10000]     # 바꿔 보세요
꺼낼_수들 = [4, 16, 32]              # 바꿔 보세요

print(f"{'상자 크기':>8} {'꺼낼 수':>6} {'도착한 판':>8} {'다 배운 두뇌':>10} {'다시 배운 횟수':>12}")
for 크기 in 상자_크기들:
    for 몇개 in 꺼낼_수들:
        결과 = 학습(판수=300, 기억_상자=기억상자(크기), 꺼낼_수=몇개, 사진_간격=10_000)
        print(f"{크기:>8} {몇개:>6} {결과['성공들'].sum():>8.0f} {다_배운_두뇌_시험(결과['Q']):>10} {결과['다시배운_횟수']:>12,}")

## 13. 🌗 좋은 기억만 · 나쁜 기억만

사람도 그렇죠. **잘된 일만 떠올리는 사람**, **실패만 곱씹는 사람**, **둘 다 돌아보는 사람.** 두뇌에게도 똑같이 시켜 봅니다.

- 한 판이 끝나면, 그 판의 SARS 를 **G 에 도착한 판(좋은 기억)** 인지 **실패한 판(나쁜 기억)** 인지에 따라 상자에 넣을지 정합니다.
- 그리고 세상을 하나 더 만듭니다: **구멍에 빠지면 −1점** (실패가 아프게 느껴지는 세상). 원래 얼음 호수는 구멍이 0점이라 「실패했다」는 신호가 없거든요.

In [ ]:
def 학습_실험(판수=500, 구멍_벌점=0.0, 남길_기억="전부", 왜곡=None, 꺼낼_수=32, seed=SEED):
    # 남길_기억: "전부" · "좋은 기억만"(G 도착한 판) · "나쁜 기억만"(실패한 판)
    # 왜곡: 실제 보상 r 을 받아 「기억에 적을 보상」을 돌려주는 함수 (None 이면 정직하게)
    rng = random.Random(seed)
    env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
    Q = np.zeros((칸수, 4))
    상자 = deque(maxlen=10000)
    엡실론, 줄이기 = 1.0, 0.01 ** (1 / (판수 * 0.6))
    도착, 구멍 = 0, 0
    for 판 in range(판수):
        s, _ = env.reset()
        끝, 이번_판 = False, []
        while not 끝:
            a = 행동_고르기(Q, s, 엡실론, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            끝 = terminated or truncated
            if terminated and r == 0:
                r = 구멍_벌점                         # 구멍에 빠짐
                구멍 += 1
            도착 += r > 0
            기억할_보상 = 왜곡(r, rng) if 왜곡 else r     # 기억에 적는 보상
            이번_판.append((s, a, 기억할_보상, s2, terminated))
            if len(상자) >= 꺼낼_수:
                for 기억 in rng.sample(상자, 꺼낼_수):
                    Q_고치기(Q, *기억)
            s = s2
        좋은_판 = any(m[2] > 0 for m in 이번_판) if not 왜곡 else (s == 칸수 - 1)
        if 남길_기억 == "전부" or (남길_기억 == "좋은 기억만" and 좋은_판) or (남길_기억 == "나쁜 기억만" and not 좋은_판):
            상자.extend(이번_판)
        엡실론 = max(0.01, 엡실론 * 줄이기)
    env.close()
    return dict(Q=Q, 도착률=도착 / 판수, 구멍률=구멍 / 판수)

def 씨앗_5개(**설정):
    결과들 = [학습_실험(seed=sd, **설정) for sd in [0, 1, 2, 3, 4]]
    return (np.mean([r["도착률"] for r in 결과들]), np.mean([r["구멍률"] for r in 결과들]),
            sum(다_배운_두뇌_시험(r["Q"]).startswith("도착") for r in 결과들))

print(f"{'':16} {'구멍 0점 (원래)':>16}   {'구멍 −1점 (실패가 아픈 세상)':>24}")
삶 = {}
for 남길 in ["전부", "좋은 기억만", "나쁜 기억만"]:
    a = 씨앗_5개(남길_기억=남길, 구멍_벌점=0.0)
    b = 씨앗_5개(남길_기억=남길, 구멍_벌점=-1.0)
    삶[남길] = (a, b)
    print(f"{남길:14} 도착률 {a[0]:5.1%} · 다 배운 두뇌 {a[2]}/5    도착률 {b[0]:5.1%} · 다 배운 두뇌 {b[2]}/5")

**읽는 법**
- **좋은 기억만** — 성공한 길은 압니다. 하지만 구멍이 어디인지는 한 번도 배운 적이 없어요. 그래서 실패가 아픈 세상에서도 더 나아지지 않습니다.
- **나쁜 기억만** — 조심하는 법만 배웁니다. 구멍은 피하지만 G 가 어디인지는 끝까지 모릅니다. 원래 호수(구멍 0점)에서는 실패가 실패로 느껴지지도 않아 아무것도 못 배워요.
- **둘 다 + 실패를 실패로 느낄 때** 가장 잘 배웁니다.

## 14. 🪞 기억을 내 맘대로 바꾸면? (왜곡)

세상은 그대로입니다 (구멍 −1, G +1). 바뀌는 건 **상자에 적는 기억**뿐.
「그 실패는 사실 괜찮았어」 「아팠던 건 잊자」 「조금 지어내도 되겠지」 — 두뇌는 어떻게 될까요?

In [ ]:
왜곡들 = {
    "① 정직하게":               None,
    "② 아픔을 잊음 (구멍→0)":     lambda r, rng: 0.0 if r < 0 else r,
    "③ 실패를 미화 (구멍→+0.5)":  lambda r, rng: 0.5 if r < 0 else r,
    "④ 공포를 과장 (구멍→−10)":   lambda r, rng: -10.0 if r < 0 else r,
    "⑤ 성공을 부풀림 (G→+10)":    lambda r, rng: 10.0 if r > 0 else r,
    "⑥ 10%는 지어냄":            lambda r, rng: rng.choice([-1.0, 0.0, 1.0]) if rng.random() < 0.1 else r,
}
거울 = {}
for 이름, 왜곡 in 왜곡들.items():
    도착률, 구멍률, 다배운 = 씨앗_5개(구멍_벌점=-1.0, 왜곡=왜곡)
    거울[이름] = (도착률, 구멍률)
    print(f"{이름:22} 도착률 {도착률:5.1%} · 구멍에 빠진 비율 {구멍률:5.1%} · 다 배운 두뇌 도착 {다배운}/5")

이름들 = list(거울)
fig, ax = plt.subplots(figsize=(10, 3.8))
y = np.arange(len(이름들))[::-1]
ax.barh(y + 0.18, [거울[n][0] for n in 이름들], height=0.36, color="#2ca02c", label="reached G")
ax.barh(y - 0.18, [거울[n][1] for n in 이름들], height=0.36, color="#d62728", label="fell in a hole")
ax.set_yticks(y); ax.set_yticklabels(["1 honest", "2 forget pain", "3 glorify failure", "4 exaggerate fear", "5 inflate success", "6 make up 10%"])
ax.set_xlim(0, 1.05); ax.legend(loc="lower right"); ax.set_xlabel("rate during 500 episodes (5 seeds avg)")
plt.show()

**읽는 법**
- **③ 실패를 미화하면** 두뇌는 구멍을 「좋은 곳」으로 기억합니다. 그래서 일부러 구멍으로 뛰어듭니다.
- **② 아픔을 잊으면** 실패에서 배우지 못해 구멍에 계속 빠집니다.
- **⑥ 기억을 10% 만 지어내도** 있지도 않은 보상을 쫓다가 목표를 잃습니다.
- **④ 과장 · ⑤ 부풀림**은 이 호수에서는 정직과 결과가 같습니다. 무엇이 더 좋은지의 **순서**가 그대로라서 고르는 행동이 바뀌지 않아요. 순서를 뒤집는 왜곡(③)이 가장 위험합니다.

## 정리

1. **행동가치 Q(s, a)** — 「이 칸에서 이 행동을 하면 앞으로 얼마나 받을까」에 대한 두뇌의 추측. 보상은 G 에서만 나오지만, 그 값이 한 칸씩 **거꾸로** 번져 출발점까지 전해집니다.
2. **SARS 쌓기** — 겪은 일(S, A, R, S')을 버리지 않고 상자에 쌓는다. 가득 차면 가장 오래된 기억부터 지운다. 처음 것만 붙들고 있으면 새로 배울 게 없다.
3. **꺼내서 다시 배우기** — 같은 경험으로 여러 번 배우니 **같은 판수로도 훨씬 빨리** 가치가 퍼진다. 어떻게 꺼내느냐(무작위 · 최근 · 보상 우선)에 따라 결과가 달라진다.
4. **대가** — 다시 배우는 만큼 계산이 늘어난다. 상자 크기 · 꺼내는 수 · 판수의 균형이 중요하다.
5. **정직한 기억** — 좋은 기억과 나쁜 기억을 함께, 있는 그대로. 실패를 미화하거나 지어내면 두뇌는 엉뚱한 곳으로 갑니다. 사람도 마찬가지죠.

🏆 **강화학습 컵 #2 · 기억하기** 에서 「기억을 가장 잘 쌓는 숫자」를 찾아 겨뤄 보세요 → https://www.aicitybuilders.com/rl-cup2

**다음 4강** — 지금 Q 표는 칸이 64개라 표로 충분합니다. 벽돌깨기처럼 화면 전체가 상태라면 칸이 셀 수 없이 많아집니다. 그래서 표 대신 **딥러닝**이 Q 를 계산합니다. 그게 DQN 이고, DQN 도 오늘 만든 기억 상자를 그대로 씁니다.